In [1]:
# 🔹 1. Imports
import torch
import torch.nn as nn
from torch.nn.utils.rnn import pad_sequence

In [2]:
# 🔹 2. Data
texts = [
    "i love this movie",
    "this is amazing",
    "i hate this",
    "this is terrible"
]

In [3]:
labels = [1, 1, 0, 0]  # 1 = positive, 0 = negative

In [4]:
# 🔹 6. Labels tensor
labels = torch.tensor(labels, dtype=torch.float32).unsqueeze(1)

In [5]:
# 🔹 3. Build Vocabulary (word → index)
all_words = " ".join(texts).split()
vocab = {word: i+1 for i, word in enumerate(set(all_words))}

In [6]:
# 🔹 4. Encode sentences (text → numbers)
def encode(sentence):
    return [vocab[word] for word in sentence.split()]

encoded_texts = [torch.tensor(encode(t)) for t in texts]

In [7]:
# 🔹 5. Pad sequences (make same length)
padded = pad_sequence(encoded_texts, batch_first=True)

In [8]:
# 🔹 7. Define RNN Model
class RNNModel(nn.Module):
    def __init__(self, vocab_size, embed_dim, hidden_dim):
        super().__init__()

        self.embedding = nn.Embedding(vocab_size, embed_dim)
        self.rnn = nn.RNN(embed_dim, hidden_dim, batch_first=True)
        self.fc = nn.Linear(hidden_dim, 1)
        self.sigmoid = nn.Sigmoid()

    def forward(self, x):
        x = self.embedding(x)           # (batch, seq_len → vectors)
        output, hidden = self.rnn(x)    # process sequence
        out = self.fc(hidden[-1])       # final memory
        return self.sigmoid(out)

In [9]:
# 🔹 8. Initialize Model
model = RNNModel(vocab_size=len(vocab)+1, embed_dim=8, hidden_dim=16)

In [ ]:
# 🔹 9. Loss & Optimizer
criterion = nn.BCELoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.01)

In [ ]:
# 🔹 10. Training Loop
for epoch in range(100):
    optimizer.zero_grad()

    outputs = model(padded)
    loss = criterion(outputs, labels)

    loss.backward()
    optimizer.step()

    if epoch % 20 == 0:
        print(f"Epoch {epoch}, Loss: {loss.item()}")

In [ ]:
# 🔹 11. Test Prediction
def predict(sentence):
    encoded = torch.tensor(encode(sentence)).unsqueeze(0)
    output = model(encoded)
    return "Positive" if output.item() > 0.5 else "Negative"

print(predict("i love this"))
print(predict("i hate this"))